![Texto alternativo](https://github.com/evmpython/Minicurso_queimadas_UNIFEI_INPE_2026/blob/main/04_logos/banner_queimadas.png?raw=true)

🦖**Título sugerido:** Sazonalidade dos focos de queimadas utilizando K-Means

---

🎯**Objetivo:**


Implementar o algoritmo K-Means para classificar os meses em níveis de baixa, média e alta atividade de queimadas no estado do Pará, utilizando a série temporal do número mensal de focos registrada entre janeiro de 2003 e julho de 2026.

---

❓ **Pergunta:**

O algoritmo não supervisionado K-Means é capaz de identificar e classificar os padrões sazonais da atividade de queimadas no estado do Pará?

---

> ⚠️ **Atenção**
>
> O termo **não supervisionado** **não** significa que o método faz previsão. Ele apenas identifica padrões e agrupamentos nos dados sem utilizar rótulos previamente conhecidos.

---


🧑**Palestrante/Tutor**

Dr. Guilherme Martins - Nottus Meteorologia

✉ guilherme.martins@nottus.com.br | jgmsantos@gmail.com

🌎 https://github.com/jgmsantos

🌎 https://guilherme.readthedocs.io/en/latest/


---


✅**Aulas ministradas**

- Aula_1_ex01: https://colab.research.google.com/drive/1mznslPhwRJlc75Vc2oJM-JR0g3M0K9b5?usp=sharing
- Aula_1_ex02: https://colab.research.google.com/drive/1Xehetg_R3WMwq1E889q7XgKbLAceur6s?usp=sharing
- Aula_1_ex03: https://colab.research.google.com/drive/1vodKYTBTvDt_htd2jpWTYc4uJDgPwhCM?usp=sharing

---

📚**Material de apoio sobre Python**

https://guilherme.readthedocs.io/en/latest/pages/tutoriais/python.html

---

🎲**Dados utilizados no formato csv**
- Focos mensais:
  - Focos de queimadadas disponibilizados de forma gratuita pelo Programa Queimadas do INPE.
  - Série temporal do acumulado mensal de focos de queimadas do satélite AQUA_M-T desde janeiro/2003 até julho/2026.

---

✅**Atividades a serem desenvolvidas**
1. Importação de bibliotecas
1. Abertura do arquivo csv
1. Análise exploratória
   - Estatística descritiva
   - Assimetria
   - Curtose
1. Distribuição dos dados (histograma)
1. Transformação da variável
1. Escolha do melhor k
   - Algumas considerações
   - Validação visual
1. Modelo K-MEANS
   - Renomear os clusters
   - Estatística dos clusters
   - Série temporal dos clusters
1. Análise estatística dos clusters
   - Frequência dos grupos
   - Tabela de contingência
1. Conclusão


---

❗**Importante**

- Necessário possuir uma conta do Gmail.
- Salvar este código no seu Google Drive. Basta clicar em **Arquivo** (canto superior esquerdo) e depois em **Salvar uma cópia no drive** e fazer o login numa conta Google.
---

In [ ]:
# Uninstall cuml to prevent GPU-related errors and ensure sklearn.cluster.KMeans uses the CPU.
#!pip uninstall -y cuml

# After running this cell, please restart the Colab runtime (Runtime -> Restart runtime) for the changes to take effect.

## Importação de bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os

from sklearn.cluster import KMeans # Algoritmo K-Means.
from sklearn.metrics import silhouette_score # Cálculo do silhouette_score.

# Montar o drive para ler/salvar arquivos que estão no seu Google Drive.
# Não precisa alterar nada, é assim mesmo.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Local para salvar as figuras.
# ALTERAR AQUI APONTANDO PARA A PASTA "figuras".
diretorio_figuras = "/content/drive/MyDrive/cursos/queimadas_2026/scripts/figuras"

# Cria o diretório acima caso ele não exista.
os.makedirs(diretorio_figuras, exist_ok=True)

## Abertura do arquivo

In [ ]:
# Abertura do arquivo.
df = pd.read_csv("/content/drive/MyDrive/cursos/queimadas_2026/scripts/figuras/acumulado_mensal.csv")

# Renomeia de Estado para  Focos.
df = df.rename(columns={"Estado": "Focos"})

# Convert "Data" column to datetime objects
df["Data"] = pd.to_datetime(df["Data"])

# Visualiza o DataFrame.
df

In [ ]:
df.info()

## Análise exploratória

Estatística descritiva dos dados mensais de focos.

In [ ]:
# Estatística descritiva.
estatisticas = (
    df["Focos"]
    .describe()
    .rename(index={
        "count": "N",
        "mean": "Média",
        "std": "Desv. Padrão",
        "min": "Mínimo",
        "25%": "1º Quartil",
        "50%": "Mediana",
        "75%": "3º Quartil",
        "max": "Máximo"
    })
    .rename("Valor")
    .to_frame()
)

# Converte os valores para inteiro.
estatisticas["Valor"] = estatisticas["Valor"].round().astype(int)

# Visualiza o DataFramel.
estatisticas

Assimetria e curtose dos dados

**Assimetria (Skewness)**

A assimetria indica se a distribuição é simétrica ou se possui uma cauda mais longa para um dos lados.

- Assimetria = 0 → distribuição aproximadamente simétrica.
- Assimetria > 0 → cauda à direita (valores altos são menos frequentes, mas existem).
- Assimetria < 0 → cauda à esquerda.

**Curtose (Kurtosis)**

A curtose mede o peso das caudas da distribuição (e, indiretamente, a presença de valores extremos).

- Curtose = 0 → semelhante à distribuição normal.
- Curtose > 0 → distribuição com caudas mais pesadas e maior ocorrência de valores extremos (leptocúrtica).
- Curtose < 0 → distribuição com caudas mais leves (platicúrtica).

In [ ]:
print(f"Assimetria: {df["Focos"].skew():.2f}")
print(f"Curtose: {df["Focos"].kurt():.2f}")

A distribuição mensal de focos é assimétrica à direita (skewness = 1,66) e leptocúrtica (kurtosis = 2,74), indicando predominância de meses com poucos focos e ocorrência ocasional de meses com valores muito elevados.

## Histograma mensal dos focos

Mostra a distribuição dos focos.

In [ ]:
plt.figure(figsize=(10,6))

plt.hist(df["Focos"], bins=30, edgecolor="black")

plt.xlabel("Número de focos")
plt.ylabel("Frequência")
plt.title("Distribuição mensal de focos")

plt.grid(alpha=0.3)

# Salva a figura
plt.savefig(f"{diretorio_figuras}/ex04-01histograma.png", dpi=300)

plt.show()

## Transformação da variável

A transformação é importante porque o K-Means utiliza distâncias euclidianas. Quando a variável apresenta forte assimetria (assimetria = 1,660), os meses com valores muito altos de focos dominam o cálculo das distâncias e acabam influenciando excessivamente a formação dos clusters.

Antes da aplicação do algoritmo K-Means, os valores mensais de focos foram transformados utilizando a função log(1+x), devido à assimetria positiva da distribuição. Essa transformação reduz a influência de valores extremos e torna as distâncias entre as observações mais equilibradas, melhorando a representatividade dos agrupamentos obtidos.

In [ ]:
# Transformação da variável focos.
X = np.log1p(df[["Focos"]])

## Escolha do melhor K

Para justificar a quantidade de cluster (agrupamento).

O código abaixo calcula a inércia e a silhueta.

Algumas considerações sobre a Silhouette:
- O índice de Silhouette não pode ser calculado para um único cluster.
- A definição do índice compara:
  - distância média entre uma observação e os demais pontos do mesmo cluster.
  - distância média entre essa observação e o cluster vizinho mais próximo.

In [ ]:
# Inicializa as listas vazias.
inercia = []
silhouette = []

# Começa em 2 porque no K-Means é necessário comparar
# com o cluster vizinho.
for k in range(2,11):

    # O K-Means escolhe os centroides iniciais de forma aleatória.
    # Esses dois parâmetros controlam como o algoritmo
    # é inicializado e a reprodutibilidade dos resultados.
    modelo = KMeans(
        n_clusters=k,
        random_state=42, # Apenas para fins de reprodutibilidade. Poderia ser qualquer número.
        n_init=20 # Define quantas inicializações independentes
                  # serão realizadas antes de selecionar a melhor solução.
                  # Faz vinte tentativas diferentes e escolhe aquela que
                  # melhor separa os grupos.
                  # o algoritmo executa o K-Means 20 vezes, cada uma com
                  # centroides iniciais diferentes.
    )

    # Treina (ajusta) o modelo K-Means aos dados (fit).
    # E prediza qual cluster pertecen cada observação (predict).
    # X são os dados mensais de focos.
    labels = modelo.fit_predict(X)

    # Armazena a inercia.
    inercia.append(modelo.inertia_)

    # Armazena a silhouette.
    silhouette.append(silhouette_score(X, labels))

In [ ]:
# Visualização do labels.
# Isso por conta do loop de 2 a 9.
labels

In [ ]:
# Armazena os resultados da inercia e silhouette em um DataFrame.
resultado = pd.DataFrame({
    "k": range(2, 11),
    "Inércia": inercia,
    "Silhouette": silhouette
})

# Os resultados com 3 casas decimais.
resultado["Inércia"] = resultado["Inércia"].round(3)
resultado["Silhouette"] = resultado["Silhouette"].round(3)

# Visualiza o DataFrame.
resultado

### Algumas considerações

A maior queda ocorre de 2 para 3 clusters, indicando que adicionar um terceiro cluster melhora bastante o agrupamento. A partir daí, os ganhos tornam-se progressivamente menores.

O número ideal de cluster seria k = 3.

**Justificativa de usar k=3:**

Apesar do maior índice de Silhouette ter sido obtido para k=2 (0,660), a solução com k=3 apresentou desempenho semelhante (0,651).

A análise da redução da inércia indicou uma queda expressiva entre k=2 e k=3, enquanto aumentos posteriores no número de clusters produziram ganhos marginais.

Dessa forma, adotou-se k=3, permitindo a classificação dos meses em três níveis de atividade de queimadas: baixa, média e alta.

> **⚠️ Importante**
>
> Embora o índice de Silhouette indique que dois clusters fornecem a melhor separação matemática, optou-se por utilizar três clusters por apresentarem uma interpretação física mais adequada, permitindo classificar os meses em baixa, média e alta atividade de queimadas.

### Validação visual

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))

ax[0].plot(range(2,11), inercia, "o-")
ax[0].set_title("Método Elbow (Cotovelo)")

ax[1].plot(range(2,11), silhouette, "o-")
ax[1].set_title("Silhouette")

plt.savefig(f"{diretorio_figuras}/ex04-02elbow_shihlouette.png", dpi=300)

plt.show()

## Modelo K-MEANS

Agora sim, usamos k=3 a partir da análise anterior.

In [ ]:
# Instanciar o modelo.
modelo = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=20
)

# Executa o K-Means nos dados (fit_predict(X)).
# Cria uma nova coluna chamada "Cluster" no seu
# DataFrame com o grupo atribuído a cada mês.
df["Cluster"] = modelo.fit_predict(X)

# Classes de cluster.
# Os rótulos não possuem nenhum significado físico.
# O K-Means não sabe isso. Precisamos olhar os centroides.
df["Cluster"].unique()

In [ ]:
# Visualização do DataFrame com o seu respectivo cluster.
df.head(20)

### Renomear os clusters

Para cada rótulo do cluster atribuir uma classe (baixa, média ou alta).

In [ ]:
# Os centroides estão na escala logarítmica.
centros = modelo.cluster_centers_.flatten()
# Retorna:              array([8.76115741, 3.68728219, 6.13364565])
# índice/Rótulos do K-MEANS         0            1            2
# Classificação                    alta        baixa        média
# Neste caso, o cluster e o índice estão na mesma posição.

# Centróides na escala original dos focos.
# centros_focos = np.expm1(centros)
# print(centros_focos) # [6380.49332218   38.93616032  460.11416202]

# Ordem dos centroides.
ordem = np.argsort(centros)

# argsort(): retorna o ÍNDICE da matriz ordenada (centros), ou seja,
# do menor para o maior.

# Retorna o índice: array([1, 2, 0])
# O menor centroide está na índice 1.
# O centroide intermediário está na índice 2.
# O maior centroide está na índice 0.

# Em termos de índice 0  1  2
#                     |  |  |
#                     |  |  Alta
#                     |  Média
#                     Baixa

# Ordem definida das classes.
mapa = {
    ordem[0]: "Baixa",
    ordem[1]: "Média",
    ordem[2]: "Alta"
}

# Aplica o mapeamento para classificar em Baixa, Média ou Alta.
df["Classe"] = df["Cluster"].map(mapa)

# Visualiza o DataFrame.
df

### Estatística dos clusters

In [ ]:
# Estatística por classe (baixa, média e alta).
estatisticas_cluster = (
    df.groupby("Classe")["Focos"]
    .agg(
        [
            "count",
            "mean",
            "median",
            "min",
            "max",
            "std"
        ]
    )
    .rename(columns={
        "count": "N",
        "mean": "Média",
        "median": "Mediana",
        "min": "Mínimo",
        "max": "Máximo",
        "std": "Desv. Padrão"
    })
)

# Transforma para valores inteiros.
estatisticas_cluster = estatisticas_cluster.round(0).astype(int)

# Visualiza o DataFrame.
estatisticas_cluster

### Série temporal dos clusters

In [ ]:
plt.figure(figsize=(15,5))

cores = {
    "Baixa":"green",
    "Média":"orange",
    "Alta":"red"
}

plt.plot(
    df["Data"],
    df["Focos"],
    color="gray",
    alpha=.4
)

for classe in cores:

    tmp = df[df["Classe"]==classe]

    plt.scatter(
        tmp["Data"],
        tmp["Focos"],
        color=cores[classe],
        label=classe
    )

plt.legend()
plt.xlabel("Data")
plt.ylabel("Focos")
plt.title("Série Temporal dos Focos de Queimadas por Classe")
plt.grid(True, linestyle="--", alpha=0.6)

# Set x-axis major ticks to be at the beginning of each year
plt.gca().xaxis.set_major_locator(mdates.YearLocator())
# Format x-axis major tick labels to "YYYY-MM"
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45, ha="right") # Rotate and align labels for better visibility
plt.tight_layout() # Adjust layout to prevent labels from overlapping

plt.savefig(f"{diretorio_figuras}/ex04-03serie_temporal_cluster.png", dpi=300)

plt.show()

## Análise estatística dos clusters

### Frequência dos grupos

In [ ]:
freq = (
    df["Classe"]
    .value_counts()
    .rename_axis("Classe")
    .reset_index(name="Meses")
)

freq["Percentual"] = (
    100 *
    freq["Meses"] /
    freq["Meses"].sum()
).round(0).astype(int)

freq

### Tabela de contingência

A tabela de contingência é uma tabela que **resume a frequência de ocorrência** conjunta entre duas ou mais variáveis categóricas. Ela permite verificar como as categorias de uma variável se distribuem em relação às categorias de outra.

No nosso caso, é gerada uma tabela de contingência entre **Mês** (1 a 12) e **Classe** (Baixa, Média e Alta).

A tabela de contingência é uma ferramenta estatística utilizada para resumir a frequência de ocorrência entre duas variáveis categóricas. Neste estudo, ela foi empregada para analisar a distribuição das classes de atividade de queimadas (baixa, média e alta) ao longo dos meses do ano, permitindo identificar o padrão sazonal dos agrupamentos obtidos pelo algoritmo K-Means.

In [ ]:
# Cria uma coluna com o nome "Mes".
df["Mes"] = pd.to_datetime(df["Data"]).dt.month

# Gera a tabela de contingência.
pd.crosstab(df["Mes"], df["Classe"])

In [ ]:
# Tabela de contingência em porcentagem.
(pd.crosstab(df["Mes"], df["Classe"], normalize="index")*100).round(0).astype(int)

Nos meses de agosto da série histórica (2003-2026), 96% dos casos foram classificados como meses de alta atividade de queimadas, enquanto 4% foram classificados como atividade média. Não foram observados meses de agosto classificados como baixa atividade.

### Visualização

Visualização da tabela de contingência.

In [ ]:
# Frequência relativa (%)
tabela_pct = (
    pd.crosstab(
        df["Mes"],
        df["Classe"],
        normalize="index"
    ) * 100
)

tabela_pct = tabela_pct.round(0).astype(int)

# Renomear meses
meses = [
    "Jan", "Fev", "Mar", "Abr", "Mai", "Jun",
    "Jul", "Ago", "Set", "Out", "Nov", "Dez"
]

# Mapeamento dos meses do formato 1, 2, ..., 12 para Jan, Fev, ..., Dez.
tabela_pct.index = meses

# Ordena classes.
tabela_pct = tabela_pct[["Baixa", "Média", "Alta"]]

# Criar rótulos com %.
anotacoes = tabela_pct.astype(str) + "%"

plt.figure(figsize=(8, 6))

# Plot da tabela de contingência.
sns.heatmap(
    tabela_pct,
    annot=anotacoes,
    fmt="",
    cmap="Reds",
    linewidths=0.5,
    cbar_kws={"label": "Frequência (%)"}
)

plt.xlabel("Classe de atividade de queimadas")
plt.ylabel("Mês")
plt.title("Distribuição mensal das classes de atividade de queimadas - Pará")

plt.tight_layout()

plt.savefig(f"{diretorio_figuras}/ex04-04tabela_contingencia.png", dpi=300)

plt.show()

# Conclusão

A distribuição mensal das classes obtidas pelo K-Means evidenciou um padrão sazonal na atividade de queimadas no estado do Pará, com predominância de meses classificados como baixa atividade durante o período chuvoso e aumento progressivo das classes média e alta durante a estação seca, especialmente nos meses de agosto a outubro.

Esse resultado mostra que o agrupamento baseado somente na magnitude dos focos conseguiu recuperar o ciclo sazonal típico das queimadas.